<tabla align="centro">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visite el aprendizaje profundo del MIT</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab2/TF_Part1_MNIST.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png" style="padding-bottom:5px;" />Ejecutar en Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab2/TF_Part1_MNIST.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png" height="70px" style="padding-bottom:5px;"  />Ver código fuente en GitHub</a></td>
</tabla>

# Información de derechos de autor

In [ ]:
# Copyright 2026 MIT Introducción al aprendizaje profundo. Reservados todos los derechos.
# 
# Licenciado bajo la Licencia MIT. No puede utilizar este archivo excepto en cumplimiento
# con la Licencia. Uso y/o modificación de este código fuera del MIT Introducción
# al Deep Learning debe hacer referencia a:
# 
# © MIT Introducción al aprendizaje profundo
# http://intotodeeplearning.com
# 

# Laboratorio 2: Visión por Computador

# Parte 1: Clasificación de dígitos MNIST

En la primera parte de esta práctica de laboratorio, construiremos y entrenaremos una red neuronal convolucional (CNN) para clasificar dígitos escritos a mano del famoso conjunto de datos [MNIST](http://yann.lecun.com/exdb/mnist/). El conjunto de datos MNIST consta de 60.000 imágenes de entrenamiento y 10.000 imágenes de prueba. Nuestras clases son los dígitos 0-9.

Primero, descarguemos el repositorio del curso, instalemos las dependencias e importemos los paquetes relevantes que necesitaremos para esta práctica de laboratorio.

In [ ]:
# Importar Tensorflow 2.0
# !pip instalar tensorflow
import tensorflow as tf

# Paquete de introducción al aprendizaje profundo del MIT
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# otros paquetes
import matplotlib.pyplot as plt
import numpy as np
import random
from tqdm import tqdm

También instalaremos Comet. Si siguió las instrucciones del Laboratorio 1, debería tener configurada su cuenta Comet. Ingrese su clave API a continuación.

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
# TODO: ¡¡INGRESA TU CLAVE API AQUÍ!!
COMET_API_KEY = ""

# Comprobar que estamos usando una GPU, si no cambiar de tiempo de ejecución
# usando Runtime > Cambiar tipo de tiempo de ejecución > GPU
assert len(tf.config.list_physical_devices('GPU')) > 0
assert COMET_API_KEY != "", "Please insert your Comet API Key"

In [ ]:
# iniciar un primer experimento con cometas para la primera parte del laboratorio
comet_ml.init(project_name="6S191_lab2_part1_NN")
comet_model_1 = comet_ml.Experiment()

## 1.1 conjunto de datos MNIST

Descarguemos y carguemos el conjunto de datos y mostremos algunas muestras aleatorias del mismo:

In [ ]:
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = (np.expand_dims(train_images, axis=-1)/255.).astype(np.float32)
train_labels = (train_labels).astype(np.int64)
test_images = (np.expand_dims(test_images, axis=-1)/255.).astype(np.float32)
test_labels = (test_labels).astype(np.int64)

Nuestro conjunto de capacitación se compone de imágenes en escala de grises de 28x28 de dígitos escritos a mano.

Visualicemos cómo lucen algunas de estas imágenes y sus correspondientes etiquetas de entrenamiento.

In [ ]:
plt.figure(figsize=(10,10))
random_inds = np.random.choice(60000,36)
for i in range(36):
    plt.subplot(6,6,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    image_ind = random_inds[i]
    plt.imshow(np.squeeze(train_images[image_ind]), cmap=plt.cm.binary)
    plt.xlabel(train_labels[image_ind])
comet_model_1.log_figure(figure=plt)

## 1.2 Red neuronal para clasificación de dígitos escritos a mano

Primero construiremos una red neuronal simple que consta de dos capas completamente conectadas y la aplicaremos a la tarea de clasificación de dígitos. En última instancia, nuestra red generará una distribución de probabilidad entre las clases de 10 dígitos (0-9). Esta primera arquitectura que construiremos se muestra a continuación:

![alt_text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab2/img/mnist_2layers_arch.png "CNN Architecture for MNIST Classification")


### Arquitectura de red neuronal totalmente conectada
Para definir la arquitectura de esta primera red neuronal completamente conectada, usaremos una vez más la API de Keras y definiremos el modelo usando la clase [`Sequential`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential). Observe cómo usamos primero una capa [`Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten), que aplana la entrada para que pueda introducirse en el modelo.

En el siguiente bloque, definirá las capas completamente conectadas de este sencillo trabajo.

In [ ]:
def build_fc_model():
  fc_model = tf.keras.Sequential([
      # Primero defina una capa Aplanar
      tf.keras.layers.Flatten(),

      # '''TODO: Definir la función de activación para la primera capa (Densa) completamente conectada.'''
      tf.keras.layers.Dense(128, activation= '''TODO'''),

      # '''TODO: Definir la segunda capa Densa para generar las probabilidades de clasificación'''
      '''[TODO Dense layer to output classification probabilities]'''

  ])
  return fc_model

model = build_fc_model()

A medida que avancemos en la siguiente parte, es posible que desee realizar cambios en la arquitectura definida anteriormente. **Tenga en cuenta que para actualizar el modelo más adelante, deberá volver a ejecutar la celda anterior para reinicializar el modelo.**

Demos un paso atrás y pensemos en la red que acabamos de crear. La primera capa de esta red, `tf.keras.layers.Flatten`, transforma el formato de las imágenes de una matriz 2D (28 x 28 píxeles) a una matriz 1D de 28 * 28 = 784 píxeles. Puede pensar en esta capa como desapilar filas de píxeles en la imagen y alinearlas. No hay parámetros aprendidos en esta capa; solo reformatea los datos.

Una vez aplanados los píxeles, la red consta de una secuencia de dos capas `tf.keras.layers.Dense`. Estas son capas neuronales completamente conectadas. La primera capa "densa" tiene 128 nodos (o neuronas). La segunda (y última) capa (¡que ha definido!) debe devolver una serie de puntuaciones de probabilidad que suman 1. Cada nodo contiene una puntuación que indica la probabilidad de que la imagen actual pertenezca a una de las clases de dígitos escritos a mano.

¡Eso define nuestro modelo totalmente conectado!

### Compila el modelo

Antes de entrenar el modelo, necesitamos definir algunas configuraciones más. Estos se agregan durante el paso [`compile`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential#compile) del modelo:

* *Función de pérdida*: define cómo medimos la precisión del modelo durante el entrenamiento. Como se explicó en la conferencia, durante el entrenamiento queremos minimizar esta función, lo que "dirigirá" el modelo en la dirección correcta.
* *Optimizador*: define cómo se actualiza el modelo en función de los datos que ve y su función de pérdida.
* *Métricas*: aquí podemos definir las métricas utilizadas para monitorear los pasos de capacitación y prueba. En este ejemplo, veremos la *precisión*, la fracción de imágenes que están clasificadas correctamente.

Comenzaremos utilizando un optimizador de descenso de gradiente estocástico (SGD) inicializado con una tasa de aprendizaje de 0,1. Dado que estamos realizando una tarea de clasificación categórica, querremos utilizar [cross entropy loss](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/sparse_categorical_crossentropy).

Querrá experimentar tanto con la elección del optimizador como con la tasa de aprendizaje y evaluar cómo afectan la precisión del modelo entrenado.

In [ ]:
'''TODO: Experiment with different optimizers and learning rates. How do these affect
    the accuracy of the trained model? Which optimizers and/or learning rates yield
    the best performance?'''
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=1e-1),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

### Entrena el modelo

Ahora estamos listos para entrenar nuestro modelo, lo que implicará introducir los datos de entrenamiento ("train_images" y "train_labels") en el modelo y luego pedirle que aprenda las asociaciones entre imágenes y etiquetas. También necesitaremos definir el tamaño del lote y la cantidad de épocas, o iteraciones sobre el conjunto de datos MNIST, que se usarán durante el entrenamiento.

En el laboratorio 1, vimos cómo podemos usar "GradientTape" para optimizar las pérdidas y entrenar modelos con descenso de gradiente estocástico. Después de definir la configuración del modelo en el paso "compilar", también podemos realizar el entrenamiento llamando al método [`fit`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential#fit) en una instancia de la clase "Modelo". Usaremos esto para entrenar nuestro modelo totalmente conectado.


In [ ]:
# Defina el tamaño del lote y la cantidad de épocas que se utilizarán durante el entrenamiento.
BATCH_SIZE = 64
EPOCHS = 5

model.fit(train_images, train_labels, batch_size=BATCH_SIZE, epochs=EPOCHS)
comet_model_1.end()

A medida que el modelo se entrena, se muestran las métricas de pérdida y precisión. Con cinco épocas y una tasa de aprendizaje de 0,01, este modelo completamente conectado debería lograr una precisión de aproximadamente 0,97 (o 97%) en los datos de entrenamiento.

### Evaluar la precisión en el conjunto de datos de prueba

Ahora que hemos entrenado el modelo, podemos pedirle que haga predicciones sobre un conjunto de pruebas que no haya visto antes. En este ejemplo, la matriz `test_images` comprende nuestro conjunto de datos de prueba. Para evaluar la precisión, podemos verificar si las predicciones del modelo coinciden con las etiquetas de la matriz `test_labels`.

Utilice el método [`evaluate`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential#evaluate) para evaluar el modelo en el conjunto de datos de prueba.

In [ ]:
'''TODO: Use the evaluate method to test the model!'''
test_loss, test_acc = # TODO

print('Precisión de la prueba:', test_acc)

Puede observar que la precisión del conjunto de datos de prueba es un poco menor que la precisión del conjunto de datos de entrenamiento. Esta brecha entre la precisión del entrenamiento y la precisión de las pruebas es un ejemplo de *sobreajuste*, cuando un modelo de aprendizaje automático funciona peor con datos nuevos que con sus datos de entrenamiento.

¿Cuál es la mayor precisión que puede lograr con este primer modelo totalmente conectado? Dado que la tarea de clasificación de dígitos escritos a mano es bastante sencilla, es posible que se pregunte cómo podemos hacerlo mejor...

![Deeper...](https://i.kym-cdn.com/photos/images/newsfeed/000/534/153/f87.jpg)

## 1.3 Red neuronal convolucional (CNN) para clasificación de dígitos escritos a mano

Como vimos en la conferencia, las redes neuronales convolucionales (CNN) son particularmente adecuadas para una variedad de tareas en visión por computadora y han logrado precisiones casi perfectas en el conjunto de datos MNIST. Ahora construiremos una CNN compuesta por dos capas convolucionales y capas de agrupación, seguidas de dos capas completamente conectadas y, en última instancia, generaremos una distribución de probabilidad entre las clases de 10 dígitos (0-9). La CNN que construiremos se muestra a continuación:

![alt_text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab2/img/convnet_fig.png "CNN Architecture for MNIST Classification")

### Definir el modelo CNN

Usaremos los mismos conjuntos de datos de entrenamiento y prueba que antes, y procederemos de manera similar a nuestra red completamente conectada para definir y entrenar nuestro nuevo modelo CNN. Para hacer esto, exploraremos dos capas que no hemos encontrado antes: puede usar [`keras.layers.Conv2D` ](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) para definir capas convolucionales y [`keras.layers.MaxPool2D`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MaxPool2D) para definir las capas de agrupación. Utilice los parámetros que se muestran en la arquitectura de red anterior para definir estas capas y construir el modelo CNN.

In [ ]:
def build_cnn_model():
    cnn_model = tf.keras.Sequential([

        # TODO: Definir la primera capa convolucional
        tf.keras.layers.Conv2D('''TODO''')

        # TODO: Definir la primera capa de agrupación máxima
        tf.keras.layers.MaxPool2D('''TODO''')

        # TODO: Definir la segunda capa convolucional
        tf.keras.layers.Conv2D('''TODO''')

        # TODO: Definir la segunda capa de agrupación máxima
        tf.keras.layers.MaxPool2D('''TODO''')

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation=tf.nn.relu),

        # TODO: Definir la última capa Densa para generar la clasificación
        # probabilidades. Preste atención a la activación necesaria una probabilidad.
        # producción
        '''[TODO Dense layer to output classification probabilities]'''
    ])

    return cnn_model

cnn_model = build_cnn_model()
# Inicialice el modelo pasando algunos datos a través de
cnn_model.predict(train_images[[0]])
# Imprime el resumen de las capas del modelo.
print(cnn_model.summary())

### Entrene y pruebe el modelo CNN

Ahora, como antes, podemos definir la función de pérdida, el optimizador y las métricas mediante el método "compilar". Compile el modelo CNN con un optimizador y una tasa de aprendizaje de su elección:

In [ ]:
comet_ml.init(project_name="6.s191lab2_part1_CNN")
comet_model_2 = comet_ml.Experiment()

'''TODO: Define the compile operation with your optimizer and learning rate of choice'''
cnn_model.compile(optimizer='''TODO''', loss='''TODO''', metrics=['accuracy']) # TODO

Como fue el caso con el modelo completamente conectado, podemos entrenar nuestra CNN usando el método "fit" a través de la API de Keras.

In [ ]:
'''TODO: Use model.fit to train the CNN model, with the same batch_size and number of epochs previously used.'''
cnn_model.fit('''TODO''')
# cometa_modelo_2.end()

¡Excelente! Ahora que hemos entrenado el modelo, evaluémoslo en el conjunto de datos de prueba usando el método [`evaluate`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential#evaluate):

In [ ]:
'''TODO: Use the evaluate method to test the model!'''
test_loss, test_acc = # TODO

print('Precisión de la prueba:', test_acc)

¿Cuál es la precisión más alta que puede lograr utilizando el modelo CNN y cómo se compara la precisión del modelo CNN con la precisión de la red simple completamente conectada? ¿Qué optimizadores y tasas de aprendizaje parecen óptimos para entrenar el modelo CNN?

No dude en hacer clic en los enlaces de Comet para investigar las curvas de entrenamiento/precisión de su modelo.

### Hacer predicciones con el modelo CNN

Con el modelo entrenado, podemos usarlo para hacer predicciones sobre algunas imágenes. La llamada a la función [`predict`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential#predict) genera las predicciones de salida dado un conjunto de muestras de entrada.


In [ ]:
predictions = cnn_model.predict(test_images)

Con esta llamada a función, el modelo ha predicho la etiqueta para cada imagen en el conjunto de prueba. Echemos un vistazo a la predicción de la primera imagen del conjunto de datos de prueba:

In [ ]:
predictions[0]

Como puede ver, una predicción es una matriz de 10 números. Recuerde que el resultado de nuestro modelo es una distribución de probabilidad entre las clases de 10 dígitos. Así, estos números describen la "confianza" del modelo en que la imagen corresponde a cada uno de los 10 dígitos diferentes.

Veamos el dígito que tiene la mayor confianza para la primera imagen en el conjunto de datos de prueba:

In [ ]:
'''TODO: identify the digit with the highest confidence prediction for the first
    image in the test dataset. '''
prediction = # TODO

print(prediction)

Entonces, el modelo está más seguro de que esta imagen es un "???". Podemos comprobar la etiqueta de prueba (recuerde, esta es la verdadera identidad del dígito) para ver si esta predicción es correcta:

In [ ]:
print("La etiqueta de este dígito es:", test_labels[0])
plt.imshow(test_images[0,:,:,0], cmap=plt.cm.binary)
comet_model_2.log_figure(figure=plt)

¡Es! Visualicemos los resultados de la clasificación en el conjunto de datos MNIST. Trazaremos imágenes del conjunto de datos de prueba junto con su etiqueta predicha, así como un histograma que proporciona las probabilidades de predicción para cada uno de los dígitos:

In [ ]:
# @title ¡Cambie el control deslizante para ver las predicciones del modelo! {ejecutar: "automático" }

image_index = 79 # @param {tipo:"control deslizante", min:0, max:100, paso:1}
plt.subplot(1,2,1)
mdl.lab2.plot_image_prediction(image_index, predictions, test_labels, test_images)
plt.subplot(1,2,2)
mdl.lab2.plot_value_prediction(image_index, predictions,  test_labels)
comet_model_2.log_figure(figure=plt)

También podemos trazar varias imágenes junto con sus predicciones, donde las etiquetas de predicción correcta son azules y las etiquetas de predicción incorrecta son grises. El número proporciona el porcentaje de confianza (sobre 100) de la etiqueta prevista. Tenga en cuenta que el modelo puede tener mucha confianza en una predicción incorrecta.

In [ ]:
# Traza las primeras X imágenes de prueba, su etiqueta predicha y la etiqueta verdadera
# Colorea las predicciones correctas en azul y las incorrectas en rojo.
num_rows = 5
num_cols = 4
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  mdl.lab2.plot_image_prediction(i, predictions, test_labels, test_images)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  mdl.lab2.plot_value_prediction(i, predictions, test_labels)
comet_model_2.log_figure(figure=plt)
comet_model_2.end()


## 1.4 Entrenando el modelo 2.0

Anteriormente en el laboratorio, utilizamos la llamada a la función [`fit`](https://www.tensorflow.org/api_docs/python/tf/keras/models/Sequential#fit) para entrenar el modelo. Esta función es bastante intuitiva y de alto nivel, lo que resulta realmente útil para modelos más simples. Como podrá ver, esta función abstrae muchos detalles en la llamada de capacitación y tenemos menos control sobre el modelo de capacitación, lo que podría ser útil en otros contextos.

Como alternativa a esto, podemos usar la clase [`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape) para registrar operaciones de diferenciación durante el entrenamiento y luego llamar a la función [`tf.GradientTape.gradient`](https://www.tensorflow.org/api_docs/python/tf/GradientTape#gradient) para calcular los gradientes. Quizás recuerdes haber visto esto en el Laboratorio 1, Parte 1, pero echemos otro vistazo aquí.

Usaremos este marco para entrenar nuestro `cnn_model` usando un descenso de gradiente estocástico.

In [ ]:
# Reconstruir el modelo CNN
cnn_model = build_cnn_model()

batch_size = 12
loss_history = mdl.util.LossHistory(smoothing_factor=0.95) # para registrar la evolución de la pérdida
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss', scale='semilogy')
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-2) # definir nuestro optimizador

comet_ml.init(project_name="6.s191lab2_part1_CNN2")
comet_model_3 = comet_ml.Experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear() # claro si existe

for idx in tqdm(range(0, train_images.shape[0], batch_size)):
  # Primero tome un lote de datos de entrenamiento y convierta las imágenes de entrada en tensores
  (images, labels) = (train_images[idx:idx+batch_size], train_labels[idx:idx+batch_size])
  images = tf.convert_to_tensor(images, dtype=tf.float32)

  # GradientTape para registrar operaciones de diferenciación
  with tf.GradientTape() as tape:
    # '''TODO: introducir las imágenes en el modelo y obtener las predicciones'''
    logits = # TODO

    # '''TODO: calcular la pérdida categórica de entropía cruzada
    loss_value = tf.keras.backend.sparse_categorical_crossentropy('''TODO''', '''TODO''') # TODO
    comet_model_3.log_metric("loss", loss_value.numpy().mean(), step=idx)

  loss_history.append(loss_value.numpy().mean()) # agregar la pérdida al registro loss_history
  plotter.plot(loss_history.get())

  # Propagación hacia atrás
  '''TODO: Use the tape to compute the gradient against all parameters in the CNN model.
      Use cnn_model.trainable_variables to access these parameters.'''
  grads = # TODO
  optimizer.apply_gradients(zip(grads, cnn_model.trainable_variables))

comet_model_3.log_figure(figure=plt)
comet_model_3.end()


## 1.5 Conclusión
En esta parte del laboratorio, tuvo la oportunidad de jugar con diferentes clasificadores MNIST con diferentes arquitecturas (solo capas completamente conectadas, CNN) y experimentar cómo los diferentes hiperparámetros afectan la precisión (tasa de aprendizaje, etc.). La siguiente parte del laboratorio explora otra aplicación de las CNN, la detección facial y algunos inconvenientes de los sistemas de IA en aplicaciones del mundo real, como problemas de sesgo.